# 02 · Embeddings + RAG — the cohort matchmaker

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sabrinaaquino/base-batches-workshop/blob/main/notebooks/02-embeddings-and-rag.ipynb)

We'll embed the Base Batches 003 cohort descriptions and answer two questions with vector math:

1. *"Which two teams should be friends?"* (cosine similarity)
2. *"Which team is closest to my idea?"* (vanilla RAG)

You'll learn:
- How to call `/embeddings` on Venice
- How to do cosine similarity in 5 lines of NumPy
- How to wire embeddings + chat into a tiny RAG pipeline

In [ ]:
%pip install --quiet openai requests python-dotenv rich numpy scikit-learn

In [ ]:
import os, sys

# Try Colab secrets first
try:
    from google.colab import userdata  # type: ignore
    api_key = userdata.get("VENICE_API_KEY")
    os.environ["VENICE_API_KEY"] = api_key
except Exception:
    api_key = os.environ.get("VENICE_API_KEY")

if not api_key:
    from getpass import getpass
    api_key = getpass("Paste your Venice API key: ").strip()
    os.environ["VENICE_API_KEY"] = api_key

from openai import OpenAI
client = OpenAI(
    api_key=api_key,
    base_url="https://api.venice.ai/api/v1",
)
print("Connected to Venice ✔")

## 1. The cohort

Twelve teams from the [Base Batches 003 announcement](https://blog.base.org/introducing-base-batches-003-2). These are real, public descriptions.

In [ ]:
cohort = [
    {"name": "Liminal",   "desc": "Self-custodial AI-native neobank."},
    {"name": "Labs",      "desc": "Credit infrastructure for AI agents and institutions."},
    {"name": "Upshot",    "desc": "Onchain prediction markets and forecasting."},
    {"name": "Glider",    "desc": "Mobile-first onchain trading."},
    {"name": "Charms",    "desc": "AI-powered consumer onchain experiences."},
    {"name": "Vendetta",  "desc": "Stablecoin payment rails for global merchants."},
    {"name": "Forecasta", "desc": "AI agents that trade prediction markets autonomously."},
    {"name": "Pylon",     "desc": "Onchain credit scoring for DeFi lending."},
    {"name": "Stellium",  "desc": "Privacy-preserving stablecoin payroll for remote teams."},
    {"name": "Rivet",     "desc": "DeFi vaults that auto-rebalance using AI."},
    {"name": "Cascade",   "desc": "x402-native marketplace for AI tools and data."},
    {"name": "Helio",     "desc": "Consumer wallet with AI assistant for everyday onchain payments."},
]
print(f"{len(cohort)} teams loaded.")

## 2. Embed every team

One API call. Venice's embedding model returns 1024-dim vectors by default.

In [ ]:
import numpy as np

resp = client.embeddings.create(
    model="text-embedding-bge-m3",
    input=[t["desc"] for t in cohort],
)

# Pull the vectors into a (N, dim) matrix
vectors = np.array([d.embedding for d in resp.data])
print(f"Shape: {vectors.shape}  (teams, dimensions)")

## 3. Who should be friends?

Cosine similarity is just normalized dot product. Five lines.

In [ ]:
# Normalize each row
norms = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
sim = norms @ norms.T

# Mask the diagonal so a team isn't its own best friend
np.fill_diagonal(sim, -1)

# Get top 3 pairs
import itertools
pairs = []
for i, j in itertools.combinations(range(len(cohort)), 2):
    pairs.append((sim[i, j], cohort[i]["name"], cohort[j]["name"]))

pairs.sort(reverse=True)
print("Top 3 strongest matches:\n")
for score, a, b in pairs[:3]:
    print(f"  {a:<10} ↔ {b:<10}   similarity {score:.3f}")
    a_desc = next(t["desc"] for t in cohort if t["name"] == a)
    b_desc = next(t["desc"] for t in cohort if t["name"] == b)
    print(f"     - {a}: {a_desc}")
    print(f"     - {b}: {b_desc}\n")

## 4. Mini RAG: ask in English, retrieve, answer

Classic RAG loop in three steps: embed the question, find nearest docs, hand them to a chat model with the question.

In [ ]:
def search(query: str, top_k: int = 3):
    q_vec = np.array(client.embeddings.create(
        model="text-embedding-bge-m3",
        input=[query],
    ).data[0].embedding)
    q_norm = q_vec / np.linalg.norm(q_vec)
    scores = norms @ q_norm
    idx = np.argsort(scores)[::-1][:top_k]
    return [(cohort[i], float(scores[i])) for i in idx]


def ask(question: str):
    matches = search(question, top_k=3)
    context = "\n".join(f"- {m['name']}: {m['desc']}" for m, _ in matches)

    resp = client.chat.completions.create(
        model="zai-org-glm-4.7",
        messages=[
            {"role": "system", "content": "Answer based ONLY on the provided context. If unsure, say so."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ],
    )
    print(f"❓ {question}\n")
    print(f"💡 {resp.choices[0].message.content}\n")
    print("📚 Used:", ", ".join(m["name"] for m, _ in matches))


ask("I want to build something for AI agents that need credit. Which teams overlap with my idea?")

## What just happened

- One Venice embedding call vectorized all 12 cohort descriptions.
- Cosine similarity surfaced the closest pairs in milliseconds.
- A second embedding call + a chat call wired up a complete RAG pipeline in ~10 lines.

This is the entire vector-search stack. No Pinecone, no LangChain, no `pip install` zoo. Just NumPy and one provider.

**Next:** [03 · Image generation](./03-image-generation.ipynb)